In [ ]:
%reload_ext autoreload
%autoreload 2
import os
import sys
sys.path.insert(0, os.path.abspath('../src'))
import pandas as pd
import geopandas as gpd
from pathlib import Path
import matplotlib.pyplot as plt

from interpolators.TIN import TIN

import numpy as np

import warnings
# remove future warning of spline mindist since we aren't even providing it anyways
warnings.filterwarnings('ignore', message='.*mindist parameter.*', category=FutureWarning)     

In [ ]:
WORK_DIR = Path('/mnt/teamgroup/AutoDBM/Public')
MBES = WORK_DIR / 'Multibeam_Validation_Points_For_Distribution'
SBES = WORK_DIR / 'Synthetic_Singlebeam_Surveys_For_Distribution'
CRS = pd.read_csv(WORK_DIR / 'survey_CRS_list.csv')

mbes = list((WORK_DIR / 'Multibeam_Validation_Points_For_Distribution').glob('*.csv'))
sbes = list((WORK_DIR / 'Synthetic_Singlebeam_Surveys_For_Distribution').glob('*.csv'))
crs_data = WORK_DIR / 'survey_CRS_list.csv'

geojson = Path(os.getcwd()).parent / 'National_Channel_Framework_-6742069550782745925.geojson'
ncf = gpd.read_file(geojson)

In [ ]:
# TODO: Build code to read the crs_data and use that for CRS instead of relying on eHydro downloaded data

In [ ]:
# Build a boundary catalog from SurveyJob layers, then match each survey to the best boundary by SurveyId
data_pairs = {}

def _norm(s):
    return "".join(ch for ch in str(s).upper() if ch.isalnum())

def _tokens(s):
    return {t for t in str(s).upper().replace("-", "_").split("_") if t}

# 1) Read all boundary candidates once
boundary_catalog = []
for gdb_file in WORK_DIR.glob("*/*.gdb"):
    try:
        sj = gpd.read_file(gdb_file, layer="SurveyJob")
    except Exception:
        continue

    if "SurveyId" not in sj.columns or sj.empty:
        continue

    # keep one record per SurveyId (first occurrence)
    for sid, grp in sj.groupby("SurveyId", dropna=True):
        boundary_catalog.append({
            "survey_id": str(sid),
            "survey_norm": _norm(sid),
            "survey_tokens": _tokens(sid),
            "gdb_file": gdb_file,
            "boundary": grp.copy()
        })

# 2) Match each survey from CRS table to best boundary
for _, items in pd.read_csv(crs_data).iterrows():
    survey = items["Survey"]

    sbes_match = [p for p in sbes if survey in str(p)]
    mbes_match = [p for p in mbes if survey in str(p)]
    if not sbes_match or not mbes_match:
        print(f"Missing SBES/MBES file for: {survey}")
        continue

    s_norm = _norm(survey)
    s_tokens = _tokens(survey)

    # exact normalized SurveyId match first
    exact = [c for c in boundary_catalog if c["survey_norm"] == s_norm]

    if exact:
        best = exact[0]
    else:
        # fallback: token overlap score
        scored = []
        for c in boundary_catalog:
            inter = len(s_tokens & c["survey_tokens"])
            denom = max(1, len(s_tokens))
            score = inter / denom
            scored.append((score, c))
        scored.sort(key=lambda x: x[0], reverse=True)
        best = scored[0][1] if scored and scored[0][0] > 0 else None

    if best is None:
        print(f"No boundary found for: {survey}")
        continue

    boundary = best["boundary"]
    data_pairs[survey] = [
        sbes_match[0],
        mbes_match[0],
        boundary,
    ]

    print(f"{survey} -> {best['survey_id']} ({best['gdb_file'].name})")

# Bathymetric Raster Generation

In [ ]:
# specify the horizontal spatial resolution of the resulting bathymetric surface. This is in ft
# 10ft is being used here to match the resolution of the CSAT inputs
grid_res = 10.0

# True if you want the outputs of the interpolation methods will be showcased in this notebook using Matplotlib
plots = False    

# Set number of processors to use based on the number of CPUs your computer has
# NOTE: this may exceed your system memory depending on how much RAM you have
# My workstation has 24 CPUs and 96 GB of RAM; I will use between 8 and 12 processors
processors = 20

In [ ]:
survey = "CENWK_DIS_MO_02_LMR_20250520_CS_181_181_8_SORT"

In [ ]:


test_pair = data_pairs[survey]
sbes_data = pd.read_csv(test_pair[0])
mbes_data = pd.read_csv(test_pair[1])
boundary = test_pair[2]
data_crs = boundary.crs

sbes_gdf = gpd.GeoDataFrame(
        sbes_data.copy(),
        geometry=gpd.points_from_xy(sbes_data["X_ft"], sbes_data["Y_ft"]),
        crs=data_crs
    )

mbes_gdf = gpd.GeoDataFrame(
        mbes_data.copy(),
        geometry=gpd.points_from_xy(mbes_data["X_ft"], mbes_data["Y_ft"]),
        crs=data_crs
    )

OUT_DIR = WORK_DIR / survey
os.makedirs(OUT_DIR, exist_ok=True)

sbes_x = sbes_gdf.X_ft
sbes_y = sbes_gdf.Y_ft
sbes_z = sbes_gdf.Elevation_ft

mbes_x = mbes_gdf.X_ft
mbes_y = mbes_gdf.Y_ft
mbes_z = mbes_gdf.Elevation_ft

fig, ax = plt.subplots(1,2, figsize=(10, 5))

# Histogram for sbes_z
ax[0].hist(sbes_z, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax[0].set_xlabel('Elevation (ft)', fontsize=12)
ax[0].set_ylabel('Frequency', fontsize=12)
ax[0].set_title(f'SBES Elevation Distribution (n={len(sbes_z):,})', fontsize=14)
ax[0].grid(axis='y', alpha=0.3)

# Histogram for mbes_z
ax[1].hist(mbes_z, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax[1].set_xlabel('Elevation (ft)', fontsize=12)
ax[1].set_ylabel('Frequency', fontsize=12)
ax[1].set_title(f'MBES Elevation Distribution (n={len(mbes_z):,})', fontsize=14)
ax[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nSBES Elevation Statistics (n={len(sbes_z):,}):")
print(f"  Mean: {np.mean(sbes_z):.2f} ft")
print(f"  Std: {np.std(sbes_z):.2f} ft")
print(f"  Min: {np.min(sbes_z):.2f} ft")
print(f"  Max: {np.max(sbes_z):.2f} ft")

print(f"\nMBES Elevation Statistics (n={len(mbes_z):,}):")
print(f"  Mean: {np.mean(mbes_z):.2f} ft")
print(f"  Std: {np.std(mbes_z):.2f} ft")
print(f"  Min: {np.min(mbes_z):.2f} ft")
print(f"  Max: {np.max(mbes_z):.2f} ft")

## Generate a bathymetric surface with TIN using the MBES soundings. This is considered ground truth.

In [ ]:
TINinterpolator = TIN(
    surveyId = survey,
    survey_path = OUT_DIR,
    x=mbes_x, y=mbes_y, z=mbes_z,
    survey_boundary = boundary,
    echosounder = "MBES",
    grid_res=grid_res,
    plot_outputs=plots,
    n_jobs=processors
)

mbes_tin_path = TINinterpolator.generate()

## Generate surface for entire synthetic SBES dataset

In [ ]:
TINinterpolator = TIN(
    surveyId = survey,
    survey_path = OUT_DIR,
    x=sbes_x, y=sbes_y, z=sbes_z,
    survey_boundary = boundary,
    echosounder = "SBES",
    grid_res=grid_res,
    plot_outputs=plots,
    n_jobs=processors
)

sbes_tin_path = TINinterpolator.generate()

## Vary the sounding and trackline spacing to show the impact of SBES sampling structure on TIN accuracy

In [ ]:
point_spacings = [4, 8, 12, 16, 20, 50, 100]   # original is 2ft
trackline_spacings = [25, 50, 100, 200, 300, 400, 500]

# one representative (first) point per transect, with cumulative along-track distance
first_pt_track_df = sbes_gdf.drop_duplicates(subset='Transect_ID').copy()
coords = np.array([(p.x, p.y) for p in first_pt_track_df.geometry])
seg = np.r_[0, np.hypot(*np.diff(coords, axis=0).T)]
first_pt_track_df['cum_dist'] = np.cumsum(seg)

results = {}
for track_spacing in trackline_spacings:
    keep_ids = []
    keep_dists = []
    next_thresh = 0
    for _, row in first_pt_track_df.iterrows():
        if row['cum_dist'] >= next_thresh:
            keep_ids.append(row['Transect_ID'])
            keep_dists.append(row['cum_dist'])
            next_thresh += track_spacing

    # actual along-track distance to each kept transect, and gap to previous kept one
    gaps = np.diff(keep_dists)                       # realized gaps (excludes the 0 seed)
    actual_track_spacing = float(np.mean(gaps)) if len(gaps) else 0.0
    gap_map = dict(zip(keep_ids, np.r_[0, gaps]))

    track_sampled = sbes_gdf[sbes_gdf['Transect_ID'].isin(keep_ids)]

    for pt_spacing in point_spacings:
        n = int(pt_spacing / 2)
        pt_sampled = (track_sampled
                      .sort_values(['Transect_ID', 'Length_along_transect_ft'])
                      .groupby('Transect_ID', group_keys=False)
                      .apply(lambda g: g.iloc[::n])
                      .copy())
        pt_sampled['target_trackline_spacing_ft'] = track_spacing
        pt_sampled['point_spacing_ft'] = pt_spacing
        pt_sampled['actual_trackline_gap_ft'] = pt_sampled['Transect_ID'].map(gap_map)
        results[(track_spacing, pt_spacing)] = pt_sampled

        sbes_x = pt_sampled.X_ft
        sbes_y = pt_sampled.Y_ft
        sbes_z = pt_sampled.Elevation_ft

        TINinterpolator = TIN(
            surveyId = survey,
            survey_path = OUT_DIR,
            x=sbes_x, y=sbes_y, z=sbes_z,
            survey_boundary = boundary,
            echosounder = f"SBES_{track_spacing}_{pt_spacing}",
            grid_res=grid_res,
            plot_outputs=plots,
            n_jobs=processors
        )

        sbes_tin_path = TINinterpolator.generate()

# optional: combine all combinations into one frame
# all_sampled = pd.concat(results.values(), ignore_index=True)